# ATLAS EDA — K. pneumoniae + Meropenem MIC Creep
## Vivli AMR Surveillance Challenge 2026

**Dataset**: ATLAS (Pfizer/Vivli), 2004–2024  
**Pathogen**: *Klebsiella pneumoniae*  
**Antibiotic**: Meropenem (carbapenem class)  
**Target**: MIC Creep — gradual upward drift in MIC₉₀ over time  

**Sections**
1. Load & filter
2. Parse censored MICs + log₂ transform
3. Year-by-year overview table
4. MIC₉₀ trend — main result chart
5. MIC distribution by year (violin)
6. Geographic analysis
7. Age group analysis
8. Paediatric vs adult MIC₉₀ over time
9. Military proxy group
10. Specimen source breakdown
11. Carbapenemase gene prevalence over time
12. Data quality (censoring & resistance rates)


Kernel to select: Python 3.12 (mic-creep)
In terminal:

cd "/Users/anna_gurina/Desktop/KSE_Bioinformatics/AMR challenge/mic-creep-predict"
.venv312/bin/python -m jupyter_server --no-browser --port 8889

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import linregress

project_root = Path('../')
sys.path.insert(0, str(project_root / 'src'))

from data.loader import ATLASLoader
from data.preprocessor import MICPreprocessor

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (13, 5), 'figure.dpi': 100})

EUCAST_R = 8       # Meropenem R breakpoint for K. pneumoniae (mg/L) — EUCAST 2024
REPORTS  = project_root / 'reports'
REPORTS.mkdir(exist_ok=True)

: 

## 1. Load K. pneumoniae + Meropenem from ATLAS

In [ ]:
data_dir = project_root / 'data' / 'raw'
loader = ATLASLoader(data_dir)
df_raw = loader.load('Klebsiella pneumoniae', antibiotic='Meropenem')

print(f"Rows:        {len(df_raw):,}")
print(f"Years:       {df_raw['Year'].min()} – {df_raw['Year'].max()}")
print(f"Countries:   {df_raw['Country'].nunique()}")
print(f"Gender dist: {df_raw['Gender'].value_counts().to_dict()}")
print(f"\nMeropenem raw value distribution (top 20):")
print(df_raw['Meropenem'].value_counts().head(20).to_string())

## 2. Parse censored MICs + log₂ transform

ATLAS uses string-encoded censored values:  
- `">8"` → 16 mg/L (next doubling dilution)  
- `"<=0.5"` → 0.25 mg/L (half the boundary)  
- `"0.25"` / `4.0` → as-is  

Then apply log₂ for the regression model.

In [ ]:
df = df_raw.copy()

# Parse each MIC string → (numeric_value, operator)
parsed = df['Meropenem'].apply(
    lambda x: MICPreprocessor.parse_censored_mic(x) if pd.notna(x) else (None, None)
)
df['mic_value']    = parsed.apply(lambda t: t[0])
df['mic_operator'] = parsed.apply(lambda t: t[1])

# Drop unparseable / non-positive values
df = df[df['mic_value'].notna() & (df['mic_value'] > 0)].copy()

df['mic_log2']    = np.log2(df['mic_value'])
df['is_censored'] = df['mic_operator'].isin(['>', '<', '>=', '<='])
df['is_resistant'] = df['mic_value'] >= EUCAST_R

print(f"Valid rows:           {len(df):,}")
print(f"Censored entries:     {df['is_censored'].sum():,}  ({df['is_censored'].mean()*100:.1f}%)")
print(f"Resistant (≥{EUCAST_R} mg/L): {df['is_resistant'].sum():,}  ({df['is_resistant'].mean()*100:.1f}%)")
print(f"\nMIC range:  {df['mic_value'].min()} – {df['mic_value'].max()} mg/L")
print(f"Log₂ range: {df['mic_log2'].min():.1f} – {df['mic_log2'].max():.1f}")

## 3. Year-by-year overview

In [ ]:
yearly = df.groupby('Year').agg(
    n            = ('mic_value', 'count'),
    mic50        = ('mic_value', lambda x: x.quantile(0.50)),
    mic90        = ('mic_value', lambda x: x.quantile(0.90)),
    pct_resistant= ('is_resistant', 'mean'),
).reset_index()
yearly['pct_resistant'] *= 100

print(f"{'Year':>4}  {'N':>6}  {'MIC₅₀':>6}  {'MIC₉₀':>6}  {'%R':>5}")
print("-" * 36)
for _, r in yearly.iterrows():
    print(f"{int(r.Year):4d}  {int(r.n):6,}  {r.mic50:6.2f}  {r.mic90:6.2f}  {r.pct_resistant:5.1f}%")

## 4. MIC₉₀ trend — main result chart

In [ ]:
slope90, intercept90, r90, p90, _ = linregress(yearly['Year'], yearly['mic90'])
slope50, intercept50, r50, p50, _ = linregress(yearly['Year'], yearly['mic50'])

fig, ax1 = plt.subplots()

# MIC₅₀ and MIC₉₀ lines
ax1.plot(yearly['Year'], yearly['mic90'], 'o-', color='#d62728', lw=2.5, ms=6, label='MIC₉₀', zorder=3)
ax1.plot(yearly['Year'], yearly['mic50'], 's--', color='#1f77b4', lw=1.5, ms=5, label='MIC₅₀', alpha=0.8, zorder=3)

# Linear trend overlays
ax1.plot(yearly['Year'], intercept90 + slope90 * yearly['Year'],
         '-', color='#d62728', alpha=0.25, lw=2)
ax1.plot(yearly['Year'], intercept50 + slope50 * yearly['Year'],
         '-', color='#1f77b4', alpha=0.25, lw=2)

# EUCAST R breakpoint
ax1.axhline(EUCAST_R, color='black', linestyle=':', lw=1.5,
            label=f'EUCAST R breakpoint ({EUCAST_R} mg/L)')

ax1.set_yscale('log', base=2)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax1.set_ylabel('Meropenem MIC (mg/L)', fontsize=11)
ax1.set_xlabel('Year', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_title(
    f'K. pneumoniae Meropenem MIC₉₀ Trend 2004–2024\n'
    f'MIC₉₀ slope: {slope90:+.3f} mg/L/yr  (R²={r90**2:.2f}, p={p90:.2e})',
    fontsize=12, fontweight='bold'
)

# Sample-size bars on secondary axis
ax2 = ax1.twinx()
ax2.bar(yearly['Year'], yearly['n'], alpha=0.10, color='grey', width=0.7)
ax2.set_ylabel('Isolates per year', fontsize=10, color='grey')
ax2.tick_params(axis='y', labelcolor='grey')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(REPORTS / 'mic90_trend_kpneumoniae_meropenem.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"MIC₉₀ slope: {slope90:+.4f} mg/L/yr  |  R²={r90**2:.3f}  |  p={p90:.2e}")
print(f"MIC₅₀ slope: {slope50:+.4f} mg/L/yr  |  R²={r50**2:.3f}  |  p={p50:.2e}")

## 5. MIC distribution by year (violin)

In [ ]:
all_years = sorted(df['Year'].unique())
# Every 2 years to keep chart readable
plot_years = [y for y in all_years if y % 2 == 0]
df_vln = df[df['Year'].isin(plot_years)].copy()

fig, ax = plt.subplots(figsize=(16, 6))
sns.violinplot(
    data=df_vln, x='Year', y='mic_log2',
    inner='quartile', palette='RdYlGn_r', density_norm='width', ax=ax
)
ax.axhline(np.log2(EUCAST_R), color='black', linestyle='--', lw=1.5,
           label=f'R breakpoint (log₂ = {np.log2(EUCAST_R):.0f})')
ax.set_ylabel('log₂(MIC) [log₂ mg/L]', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title('Meropenem MIC Distribution by Year — K. pneumoniae (ATLAS)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(REPORTS / 'mic_violin_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Geographic analysis

In [ ]:
country_stats = (
    df.groupby('Country')
    .agg(
        n            = ('mic_value', 'count'),
        mic90        = ('mic_value', lambda x: x.quantile(0.90)),
        pct_resistant= ('is_resistant', 'mean'),
    )
    .reset_index()
)
# Keep countries with ≥50 isolates for reliability
country_stats = country_stats[country_stats['n'] >= 50].copy()
top_mic90  = country_stats.sort_values('mic90', ascending=False).head(30)
top_resist = country_stats.sort_values('pct_resistant', ascending=False).head(30)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

axes[0].barh(top_mic90['Country'], top_mic90['mic90'], color='#d62728', alpha=0.8)
axes[0].axvline(EUCAST_R, color='black', linestyle='--', lw=1.2, label='R breakpoint')
axes[0].set_xlabel('MIC₉₀ (mg/L)', fontsize=11)
axes[0].set_title('MIC₉₀ by Country (top 30, ≥50 isolates)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].barh(top_resist['Country'], top_resist['pct_resistant'] * 100, color='#ff7f0e', alpha=0.8)
axes[1].set_xlabel('% Resistant (MIC ≥ 8 mg/L)', fontsize=11)
axes[1].set_title('% Resistant by Country (top 30)', fontsize=11, fontweight='bold')

plt.suptitle('K. pneumoniae Meropenem — Geographic Analysis (ATLAS)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'geo_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Age group analysis

In [ ]:
age_order = ['0 - 17', '18 - 30', '31 - 60', '61+']
age_stats = (
    df[df['Age Group'].isin(age_order)]
    .groupby('Age Group')
    .agg(
        n            = ('mic_value', 'count'),
        mic50        = ('mic_value', lambda x: x.quantile(0.50)),
        mic90        = ('mic_value', lambda x: x.quantile(0.90)),
        pct_resistant= ('is_resistant', 'mean'),
    )
    .reindex(age_order)
)
print("Age group summary:")
print(age_stats.to_string())

palette = sns.color_palette('Set2', len(age_order))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = range(len(age_stats))

axes[0].bar(x, age_stats['mic90'], color=palette)
axes[0].axhline(EUCAST_R, linestyle='--', color='black', lw=1.2, label='R breakpoint')
axes[0].set_xticks(x); axes[0].set_xticklabels(age_stats.index)
axes[0].set_ylabel('MIC₉₀ (mg/L)'); axes[0].set_title('MIC₉₀ by Age Group'); axes[0].legend(fontsize=8)

axes[1].bar(x, age_stats['pct_resistant'] * 100, color=palette)
axes[1].set_xticks(x); axes[1].set_xticklabels(age_stats.index)
axes[1].set_ylabel('% Resistant'); axes[1].set_title('% Resistant by Age Group')

plt.suptitle('K. pneumoniae Meropenem — Age Group Analysis (ATLAS)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Paediatric vs adult MIC₉₀ over time

In [ ]:
df['age_group_broad'] = 'Adult (18–60)'
df.loc[df['Age Group'] == '0 - 17', 'age_group_broad'] = 'Paediatric (0–17)'
df.loc[df['Age Group'] == '61+',    'age_group_broad'] = 'Elderly (61+)'

group_yearly = (
    df.groupby(['Year', 'age_group_broad'])['mic_value']
    .quantile(0.90)
    .reset_index()
)
group_yearly.columns = ['Year', 'Group', 'MIC90']

colors = {
    'Paediatric (0–17)': '#e377c2',
    'Adult (18–60)':     '#1f77b4',
    'Elderly (61+)':     '#ff7f0e',
}

fig, ax = plt.subplots()
for grp, gdf in group_yearly.groupby('Group'):
    ax.plot(gdf['Year'], gdf['MIC90'], 'o-', label=grp, color=colors.get(grp, 'grey'), lw=2, ms=5)
ax.axhline(EUCAST_R, color='black', linestyle=':', lw=1.5, label='EUCAST R breakpoint')
ax.set_yscale('log', base=2)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.set_ylabel('MIC₉₀ (mg/L)', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.legend(fontsize=9)
ax.set_title('MIC₉₀ by Age Group Over Time — K. pneumoniae Meropenem (ATLAS)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'mic90_by_age_group.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Military proxy group

**Definition**: wound/abscess isolates from males aged 18–60  
Proxy for combat-related infections per project design.

In [ ]:
WOUND_SOURCES = {'Wound', 'Abscess', 'Skin and Soft Tissue'}

df['military_proxy'] = (
    df['Source'].isin(WOUND_SOURCES) &
    (df['Gender'] == 'Male') &
    (df['Age Group'].isin(['18 - 30', '31 - 60']))
)

print(f"Military proxy isolates: {df['military_proxy'].sum():,}  "
      f"({df['military_proxy'].mean()*100:.1f}% of dataset)")
print(f"Source breakdown in proxy:\n{df[df['military_proxy']]['Source'].value_counts().to_string()}")

mil_yr  = df[df['military_proxy']].groupby('Year')['mic_value'].quantile(0.90).reset_index()
gen_yr  = df[~df['military_proxy']].groupby('Year')['mic_value'].quantile(0.90).reset_index()

fig, ax = plt.subplots()
ax.plot(gen_yr['Year'],  gen_yr['mic_value'],  'o-', color='#1f77b4', lw=2, ms=5, label='General population')
ax.plot(mil_yr['Year'],  mil_yr['mic_value'],  's--', color='#d62728', lw=2, ms=6, label='Military proxy (wound/male/18–60)')
ax.axhline(EUCAST_R, color='black', linestyle=':', lw=1.5, label='EUCAST R breakpoint')
ax.set_yscale('log', base=2)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
ax.set_ylabel('MIC₉₀ (mg/L)', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.legend(fontsize=9)
ax.set_title('Military Proxy vs General Population — MIC₉₀ Trend', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'mic90_military_proxy.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Specimen source breakdown

In [ ]:
source_stats = (
    df.groupby('Source')
    .agg(
        n            = ('mic_value', 'count'),
        mic90        = ('mic_value', lambda x: x.quantile(0.90)),
        pct_resistant= ('is_resistant', 'mean'),
    )
    .reset_index()
)
source_stats = source_stats[source_stats['n'] >= 100].sort_values('mic90', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
norm = plt.Normalize(source_stats['mic90'].min(), source_stats['mic90'].max())
colors_bar = plt.cm.RdYlGn_r(norm(source_stats['mic90']))
ax.barh(source_stats['Source'], source_stats['mic90'], color=colors_bar, alpha=0.9)
ax.axvline(EUCAST_R, color='black', linestyle='--', lw=1.5, label='R breakpoint')
for i, (_, row) in enumerate(source_stats.iterrows()):
    ax.text(row['mic90'] + 0.15, i, f"n={int(row['n']):,}  ({row['pct_resistant']*100:.0f}%R)",
            va='center', fontsize=8)
ax.set_xlabel('MIC₉₀ (mg/L)', fontsize=11)
ax.set_title('MIC₉₀ by Specimen Source (≥100 isolates) — K. pneumoniae Meropenem (ATLAS)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 11. Carbapenemase gene prevalence over time

ATLAS includes columns for resistance gene families detected in each isolate.  
Key carbapenemases for *K. pneumoniae*: **KPC**, **NDM**, **OXA**, **VIM**, **IMP**.

In [ ]:
CARBAPENEMASE_GENES = ['KPC', 'NDM', 'OXA', 'VIM', 'IMP', 'GES']
gene_cols = [g for g in CARBAPENEMASE_GENES if g in df.columns]

if gene_cols:
    # Presence = non-null, non-empty, non-zero string
    for gene in gene_cols:
        s = df[gene].astype(str).str.strip()
        df[f'{gene}_pos'] = df[gene].notna() & ~s.isin(['', 'nan', '0', 'None'])

    pos_cols  = [f'{g}_pos' for g in gene_cols]
    gene_yr   = df.groupby('Year')[pos_cols].mean().reset_index()
    gene_yr.columns = ['Year'] + gene_cols

    fig, ax = plt.subplots()
    for gene, color in zip(gene_cols, sns.color_palette('tab10', len(gene_cols))):
        ax.plot(gene_yr['Year'], gene_yr[gene] * 100, 'o-', label=gene, color=color, lw=2, ms=5)
    ax.set_ylabel('% Isolates with Gene', fontsize=11)
    ax.set_xlabel('Year', fontsize=11)
    ax.legend(title='Carbapenemase Gene', fontsize=9)
    ax.set_title('Carbapenemase Gene Prevalence Over Time — K. pneumoniae (ATLAS)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORTS / 'gene_prevalence_over_time.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Among resistant isolates, what fraction carry each gene?
    res_df = df[df['is_resistant']]
    print(f"\nAmong {len(res_df):,} resistant isolates (MIC ≥ {EUCAST_R} mg/L):")
    for gene in gene_cols:
        pct = res_df[f'{gene}_pos'].mean() * 100
        print(f"  {gene:<6}: {pct:.1f}% carry the gene")
else:
    print("No carbapenemase gene columns found in loaded data.")

## 12. Data quality — censoring & resistance rates over time

In [ ]:
quality = df.groupby('Year').agg(
    n_total      = ('mic_value', 'count'),
    n_censored   = ('is_censored', 'sum'),
    n_resistant  = ('is_resistant', 'sum'),
).reset_index()
quality['pct_censored']  = quality['n_censored']  / quality['n_total'] * 100
quality['pct_resistant'] = quality['n_resistant'] / quality['n_total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(quality['Year'], quality['pct_censored'], color='#aec7e8', alpha=0.9)
axes[0].set_ylabel('% Censored MIC values', fontsize=11)
axes[0].set_xlabel('Year', fontsize=11)
axes[0].set_title('Censoring Rate Over Time', fontsize=11, fontweight='bold')

axes[1].plot(quality['Year'], quality['pct_resistant'], 'o-', color='#d62728', lw=2, ms=5)
axes[1].set_ylabel('% Resistant (MIC ≥ 8 mg/L)', fontsize=11)
axes[1].set_xlabel('Year', fontsize=11)
axes[1].set_title('Resistance Rate Over Time', fontsize=11, fontweight='bold')

plt.suptitle('Data Quality & Resistance Overview — K. pneumoniae Meropenem (ATLAS)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nOverall censoring rate: {:.1f}%".format(quality['pct_censored'].mean()))
print("Overall resistance rate: {:.1f}%".format(quality['pct_resistant'].mean()))

## Summary

| Metric | Value |
|--------|-------|
| Dataset | ATLAS 2004–2024, *K. pneumoniae* + Meropenem |
| Usable isolates | see cell 2 output |
| MIC₉₀ log₂ slope | see cell 4 output |
| Primary creep signal | year (from SHAP — to be confirmed in model notebook) |

**Next step → `05_feature_engineering.ipynb`**: OHE encoding, resistance gene flags, time-split preparation.